<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week8/Day5/RAG_Databricks_Dolly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q \
    "transformers==4.49.0" \
    "sentence-transformers==3.4.1" \
    "datasets==3.3.2" \
    "huggingface-hub==0.36.2" \
    "langchain-community" \
    "langchain-huggingface" \
    "langchain-text-splitters" \
    "langchain-classic" \
    "faiss-cpu"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.12.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
%pip install -q \
    "huggingface-hub>=1.32.0,<2.0" \
    "transformers>=5.0,<6.0" \
    "sentence-transformers>=6.0,<7.0" \
    "datasets>=4.0,<6.0" \
    "requests==2.32.4" \
    "fsspec==2025.12.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:
import torch
import transformers
import datasets
import faiss

from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

print("All imports successful!")
print("GPU available:", torch.cuda.is_available())
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)

/tmp/ipykernel_9061/2816735503.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import HuggingFaceDatasetLoader


All imports successful!
GPU available: True
Transformers version: 5.20.2
Datasets version: 5.1.0


In [4]:
from datasets import load_dataset
from langchain_core.documents import Document

# Load a manageable subset of the Dolly dataset
dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train[:200]"
)

# Convert dataset rows into LangChain documents
data = [
    Document(
        page_content=row["context"] or row["response"],
        metadata={
            "instruction": row["instruction"],
            "category": row["category"]
        }
    )
    for row in dataset
]

print("Dataset records:", len(dataset))
print("LangChain documents:", len(data))
print("\nFirst document:")
print(data[0].page_content[:500])

Dataset records: 200
LangChain documents: 200

First document:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne a


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split documents into overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = text_splitter.split_documents(data)

print("Original documents:", len(data))
print("Total chunks:", len(docs))
print("\nFirst chunk:")
print(docs[0].page_content[:500])

Original documents: 200
Total chunks: 257

First chunk:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne a


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model using GPU
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": False}
)

# Test embedding generation
test_embedding = embeddings.embed_query("This is a test document.")

print("Embedding model loaded successfully!")
print("Embedding dimensions:", len(test_embedding))
print("First 3 values:", test_embedding[:3])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
Embedding dimensions: 384
First 3 values: [-0.0383385568857193, 0.12346474081277847, -0.02864294871687889]


In [7]:
from langchain_community.vectorstores import FAISS
import time

# Create the FAISS vector store
start_time = time.time()

db = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

elapsed_time = time.time() - start_time

print("FAISS vector store created successfully!")
print("Indexed chunks:", db.index.ntotal)
print(f"Processing time: {elapsed_time:.2f} seconds")

FAISS vector store created successfully!
Indexed chunks: 257
Processing time: 0.71 seconds


In [8]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Load the question-answering model and tokenizer
model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForQuestionAnswering.from_pretrained(
    model_name
).to("cuda")

print("TinyBERT model loaded successfully!")
print("Device:", next(model.parameters()).device)

Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.bias   | UNEXPECTED |  | 
fit_dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TinyBERT model loaded successfully!
Device: cuda:0


In [9]:
import torch

# Prepare question and context
question = "When did Virgin Australia commence services?"
context = docs[0].page_content

inputs = tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation="only_second",
    max_length=512
).to("cuda")

# Run extractive question answering
model.eval()

with torch.no_grad():
    outputs = model(**inputs)

# Find the predicted answer span
start_index = torch.argmax(outputs.start_logits).item()
end_index = torch.argmax(outputs.end_logits).item()

answer = tokenizer.decode(
    inputs["input_ids"][0][start_index:end_index + 1],
    skip_special_tokens=True
) if end_index >= start_index else ""

print("Question:", question)
print("Answer:", answer)

Question: When did Virgin Australia commence services?
Answer: 31 august 2000


In [10]:
# Create a FAISS retriever
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Test document retrieval
question = "When did Virgin Australia commence services?"

retrieved_docs = retriever.invoke(question)

print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nDocument {i}:")
    print(doc.page_content[:300])

Retrieved documents: 4

Document 1:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major a

Document 2:
The British Virgin Islands (BVI), officially the Virgin Islands, are a British Overseas Territory in the Caribbean, to the east of Puerto Rico and the US Virgin Islands and north-west of Anguilla. The islands are geographically part of the Virgin Islands archipelago and are located in the Leeward Is

Document 3:
St Lucia, West End, Toowong, Bulimba and Newstead border the Brisbane River. Wilston, Wooloongabba, Indooroopilly and Bowen Hills are inland and do not border the Brisbane River.

Document 4:
This stands for life assurance premium relief. Before 14 March 1984 there was tax relief on life insurance premiums paid by policyholders for po

In [11]:
def rag_answer(question, retriever, model, tokenizer):
    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Use the most relevant document as context
    context = retrieved_docs[0].page_content

    # Tokenize question and context
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=512
    ).to("cuda")

    # Generate answer span
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    start_index = torch.argmax(outputs.start_logits).item()
    end_index = torch.argmax(outputs.end_logits).item()

    answer = (
        tokenizer.decode(
            inputs["input_ids"][0][start_index:end_index + 1],
            skip_special_tokens=True
        )
        if end_index >= start_index
        else ""
    )

    return {
        "question": question,
        "answer": answer,
        "source": retrieved_docs[0].metadata
    }


print("RAG function created successfully!")

RAG function created successfully!


In [12]:
# Test the complete RAG system
question = "When did Virgin Australia commence services?"

result = rag_answer(
    question=question,
    retriever=retriever,
    model=model,
    tokenizer=tokenizer
)

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Source:", result["source"])

Question: When did Virgin Australia commence services?
Answer: 31 august 2000
Source: {'instruction': 'When did Virgin Australia start operating?', 'category': 'closed_qa'}


In [13]:
# Check whether the dataset subset contains cheesemaking information
matches = [
    doc for doc in data
    if "cheese" in doc.page_content.lower()
    or "cheese" in doc.metadata.get("instruction", "").lower()
]

print("Documents mentioning cheese:", len(matches))

for doc in matches[:3]:
    print("\nInstruction:", doc.metadata.get("instruction"))
    print("Content:", doc.page_content[:300])

Documents mentioning cheese: 0


In [14]:
from datasets import load_dataset

# Find the relevant record without downloading the dataset again
cheese_dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train",
    keep_in_memory=False
)

matches = [
    row for row in cheese_dataset
    if "cheesemaking" in row["instruction"].lower()
]

print("Matching records:", len(matches))

for row in matches[:2]:
    print("\nInstruction:", row["instruction"])
    print("Context:", row["context"][:500])
    print("Response:", row["response"][:500])

Matching records: 1

Instruction: From the provided description of the cheesemaking process, list the ingredients required to make cheese.
Context: The goal of cheese making is to control the spoiling of milk into cheese. The milk is traditionally from a cow, goat, sheep or buffalo, although, in theory, cheese could be made from the milk of any mammal. Cow's milk is most commonly used worldwide. The cheesemaker's goal is a consistent product with specific characteristics (appearance, aroma, taste, texture). The process used to make a Camembert will be similar to, but not quite the same as, that used to make Cheddar.

Some cheeses may be del
Response: The main ingredient used to make cheese is milk. Cow's milk is commonly used, though goat, sheep or buffalo can also be used, as could the milk of any mammal in theory. Starter cultures are typically added to aid in the culturing stage of the process. Rennet is added to the cheese milk to promote the separation into cheese curd and whey. U

In [15]:
from langchain_core.documents import Document

# Create a document from the cheesemaking record
cheese_row = matches[0]

cheese_document = Document(
    page_content=cheese_row["context"],
    metadata={
        "instruction": cheese_row["instruction"],
        "category": cheese_row["category"]
    }
)

# Split and add the document to the existing FAISS index
cheese_chunks = text_splitter.split_documents([cheese_document])

db.add_documents(cheese_chunks)

print("Cheesemaking chunks added:", len(cheese_chunks))
print("Total indexed chunks:", db.index.ntotal)

Cheesemaking chunks added: 9
Total indexed chunks: 266


In [16]:
# Test the required cheesemaking question
question = "What is cheesemaking?"

result = rag_answer(
    question=question,
    retriever=retriever,
    model=model,
    tokenizer=tokenizer
)

print("Question:", result["question"])
print("Answer:", result["answer"])
print("Source:", result["source"])

Question: What is cheesemaking?
Answer: to control the spoiling of milk into cheese
Source: {'instruction': 'From the provided description of the cheesemaking process, list the ingredients required to make cheese.', 'category': 'information_extraction'}


### Conclusion

In this challenge, I implemented a Retrieval-Augmented Question Answering (RAG) system using Hugging Face, LangChain, FAISS, and Intel Dynamic TinyBERT.

The system loads and processes documents from the Databricks Dolly 15k dataset, splits them into chunks, generates embeddings, and indexes them using FAISS.

Due to changes in Transformers 5, I implemented extractive question answering directly with PyTorch instead of using the deprecated question-answering pipeline.

The system successfully retrieved relevant documents and answered questions about Virgin Australia and cheesemaking.

**Result:** The RAG workflow was successfully demonstrated using semantic retrieval and extractive question answering.